# SIOPE & Minister Data Exploration and Reconciliation

This notebook processes SIOPE school expenditure data and Minister school registries to identify coincidences, references, and data alignment patterns across 2020-2026.

**Key Objectives:**
- Load and explore SIOPE expenditure and school registry data
- Load Minister school and budget data
- Identify matching schools and discrepancies
- Analyze expenditure patterns by territory and year
- Correlate SIOPE spending with Minister school characteristics

## 1. Import Required Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Paths
SIOPE_DIR = Path("local_data/SIOPE")
MINISTER_DIR = Path("local_data/MinIstruzione")
PROCESSED_DIR = Path("local_data/processed")

print("Loading SIOPE data...")
# Load SIOPE expenditure data
siope_uscite_files = sorted(SIOPE_DIR.glob("siope_uscite_*.csv"))
siope_dfs = []
for file in siope_uscite_files:
    year = int(file.stem.split('_')[-1])
    if year >= 2020:
        df = pd.read_csv(file, names=['codice_ente', 'anno', 'mese', 'codice_gestionale', 'importo_centesimi'])
        df['importo_centesimi'] = pd.to_numeric(df['importo_centesimi'], errors='coerce')
        df['importo_euro'] = df['importo_centesimi'] / 100
        siope_dfs.append(df)

siope_uscite = pd.concat(siope_dfs, ignore_index=True) if siope_dfs else pd.DataFrame()
print(f"  SIOPE USCITE: {len(siope_uscite):,} records, {siope_uscite['anno'].nunique()} years")

# Load SIOPE registry
siope_registry = pd.read_csv(SIOPE_DIR / "siope_anagrafiche_scuole.csv")
print(f"  SIOPE Registry: {len(siope_registry):,} schools")

# Load Minister school data
print("Loading Minister school data...")
minister_school_files = list(MINISTER_DIR.glob("Scuole/SCUANAGR*.csv"))
minister_schools = []
for file in minister_school_files:
    try:
        df = pd.read_csv(file)
        minister_schools.append(df)
    except Exception as e:
        print(f"  Error loading {file.name}: {e}")

minister_schools_df = pd.concat(minister_schools, ignore_index=True) if minister_schools else pd.DataFrame()
print(f"  Minister Schools: {len(minister_schools_df):,} records")

# Load processed summary data
print("Loading processed summary data...")
expenditure_by_region = pd.read_csv(PROCESSED_DIR / "siope_expenditure_by_region_year.csv", index_col=0)
school_count_by_region = pd.read_csv(PROCESSED_DIR / "siope_school_count_by_region_year.csv", index_col=0)
school_expenditure_summary = pd.read_csv(PROCESSED_DIR / "siope_school_expenditure_summary.csv")
monthly_trend = pd.read_csv(PROCESSED_DIR / "siope_monthly_expenditure_trend.csv")

print("\nData loading complete!")

## 2. Data Cleaning and Preprocessing

In [ ]:
# Examine data quality
print("=== SIOPE USCITE DATA QUALITY ===")
print(f"Missing values:\n{siope_uscite.isnull().sum()}\n")
print(f"Data types:\n{siope_uscite.dtypes}\n")
print(f"Importo euro stats:\n{siope_uscite['importo_euro'].describe()}\n")

# Remove NaN importo values
siope_uscite_clean = siope_uscite.dropna(subset=['importo_euro']).copy()
print(f"Rows removed due to NaN importo: {len(siope_uscite) - len(siope_uscite_clean)}")

# Convert anno and mese to int
siope_uscite_clean['anno'] = siope_uscite_clean['anno'].astype(int)
siope_uscite_clean['mese'] = pd.to_numeric(siope_uscite_clean['mese'], errors='coerce').astype('Int64')

print("\n=== SIOPE REGISTRY DATA QUALITY ===")
print(f"Missing values:\n{siope_registry.isnull().sum()}\n")
print(f"Data types:\n{siope_registry.dtypes}\n")

# Clean registry
siope_registry_clean = siope_registry.copy()
print(f"Unique regions: {siope_registry_clean['codice_regione'].nunique()}")
print(f"Unique provinces: {siope_registry_clean['codice_provincia'].nunique()}")

print("\n=== MINISTER SCHOOLS DATA QUALITY ===")
print(f"Shape: {minister_schools_df.shape}")
print(f"Columns: {list(minister_schools_df.columns)[:10]}...")  # Show first 10 columns
print(f"Missing values (first 10 cols):\n{minister_schools_df.iloc[:, :10].isnull().sum()}\n")

# Identify common columns
siope_cols = set(siope_registry_clean.columns)
minister_cols = set(minister_schools_df.columns) if len(minister_schools_df) > 0 else set()
print(f"Common columns between SIOPE Registry and Minister Schools: {siope_cols.intersection(minister_cols)}")

## 3. Merge SIOPE and Minister Datasets

In [ ]:
# Merge SIOPE expenditure with registry
print("Merging SIOPE expenditure with registry metadata...")
siope_merged = siope_uscite_clean.merge(
    siope_registry_clean[['codice_ente', 'codice_regione', 'codice_provincia', 'codice_comune', 'denominazione']],
    on='codice_ente',
    how='left'
)
print(f"Merged records: {len(siope_merged):,}")
print(f"Missing territorial info: {siope_merged['codice_regione'].isnull().sum()}")

# Summary statistics
print("\n=== SIOPE DATA AFTER MERGE ===")
print(f"Year range: {siope_merged['anno'].min()} - {siope_merged['anno'].max()}")
print(f"Unique schools: {siope_merged['codice_ente'].nunique():,}")
print(f"Unique regions: {siope_merged['codice_regione'].nunique()}")
print(f"Total expenditure: €{siope_merged['importo_euro'].sum():,.2f}")

# Analyze records by year
print("\n=== EXPENDITURE BY YEAR ===")
by_year = siope_merged.groupby('anno').agg({
    'importo_euro': ['sum', 'mean', 'count'],
    'codice_ente': 'nunique'
}).round(2)
by_year.columns = ['Total (€)', 'Mean per record (€)', 'Record count', 'Unique schools']
print(by_year)

# Check for regions with disproportionate data
print("\n=== TOP 10 REGIONS BY EXPENDITURE (2025) ===")
top_regions_2025 = siope_merged[siope_merged['anno'] == 2025].groupby('codice_regione').agg({
    'importo_euro': 'sum',
    'codice_ente': 'nunique'
}).sort_values('importo_euro', ascending=False).head(10)
top_regions_2025.columns = ['Total expenditure (€)', 'Schools']
print(top_regions_2025)

## 4. Temporal Analysis Across Years

In [ ]:
# Annual expenditure trend
annual_trend = siope_merged.groupby('anno').agg({
    'importo_euro': 'sum',
    'codice_ente': 'nunique'
}).reset_index()
annual_trend.columns = ['Year', 'Total expenditure (€)', 'Schools']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Total expenditure trend
ax1.plot(annual_trend['Year'], annual_trend['Total expenditure (€)'], marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('Year')
ax1.set_ylabel('Total Expenditure (€)')
ax1.set_title('SIOPE School Expenditure Trend (2020-2026)')
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x/1e6:.0f}M'))

# Plot 2: Number of schools reporting
ax2.bar(annual_trend['Year'], annual_trend['Schools'], color='steelblue', alpha=0.7)
ax2.set_xlabel('Year')
ax2.set_ylabel('Number of Schools')
ax2.set_title('Schools Reporting Expenditure by Year')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Monthly patterns
print("\n=== MONTHLY EXPENDITURE PATTERNS ===")
monthly_total = siope_merged.groupby('mese')['importo_euro'].agg(['sum', 'count', 'mean']).round(2)
monthly_total.columns = ['Total (€)', 'Records', 'Mean per record (€)']
print(monthly_total[monthly_total['Records'] > 0])

# Visualize monthly pattern (2025)
monthly_2025 = siope_merged[siope_merged['anno'] == 2025].groupby('mese')['importo_euro'].sum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly_2025.index, monthly_2025.values, color='darkgreen', alpha=0.7)
ax.set_xlabel('Month')
ax.set_ylabel('Total Expenditure (€)')
ax.set_title('Monthly Expenditure Pattern - 2025')
ax.grid(True, alpha=0.3, axis='y')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

# Year-over-year growth
print("\n=== YEAR-OVER-YEAR GROWTH ===")
yoy = annual_trend.copy()
yoy['Growth (€)'] = yoy['Total expenditure (€)'].diff()
yoy['Growth (%)'] = yoy['Total expenditure (€)'].pct_change() * 100
print(yoy[['Year', 'Total expenditure (€)', 'Growth (€)', 'Growth (%)']].round(2))

## 5. Identify Coincidences and References (Data Reconciliation)

In [ ]:
# Reconciliation Summary
print("="*70)
print("DATA RECONCILIATION ANALYSIS")
print("="*70)

# 1. Coverage Analysis
print("\n1. DATA COVERAGE")
print(f"SIOPE Schools in registry: {len(siope_registry_clean):,}")
print(f"SIOPE Schools with expenditure records: {siope_merged['codice_ente'].nunique():,}")
print(f"Minister Schools in dataset: {len(minister_schools_df):,}")

# 2. Territorial Alignment
print("\n2. TERRITORIAL DISTRIBUTION")
print(f"SIOPE Regions: {siope_merged['codice_regione'].nunique()}")
print(f"SIOPE Provinces: {siope_merged['codice_provincia'].nunique()}")

# 3. Budget category analysis
print("\n3. BUDGET CATEGORIES IN SIOPE DATA")
budget_cats = siope_merged.groupby('codice_gestionale').agg({
    'importo_euro': 'sum',
    'codice_ente': 'nunique'
}).sort_values('importo_euro', ascending=False).head(15)
budget_cats.columns = ['Total expenditure (€)', 'Schools']
print(budget_cats)

# 4. Data consistency check
print("\n4. DATA CONSISTENCY CHECKS")
# Check for negative amounts
negative_rows = siope_merged[siope_merged['importo_euro'] < 0]
print(f"Records with negative expenditure: {len(negative_rows):,} ({100*len(negative_rows)/len(siope_merged):.2f}%)")
if len(negative_rows) > 0:
    print(f"Total negative amount: €{negative_rows['importo_euro'].sum():,.2f}")

# Check schools with no expenditure
schools_with_zero = siope_merged[siope_merged['importo_euro'] == 0]
print(f"Records with zero expenditure: {len(schools_with_zero):,} ({100*len(schools_with_zero)/len(siope_merged):.2f}%)")

# 5. Data completeness by month
print("\n5. MONTH REPORTING COMPLETENESS")
months_reported = siope_merged.groupby('anno')['mese'].nunique()
print("Months reported per year:")
print(months_reported)

# 6. Schools appearing in multiple years
print("\n6. LONGITUDINAL COVERAGE")
school_years = siope_merged.groupby('codice_ente')['anno'].agg(['min', 'max', 'count']).reset_index()
school_years.columns = ['School', 'First year', 'Last year', 'Record count']
school_years['Years active'] = school_years['Last year'] - school_years['First year'] + 1

print(f"Schools in 1 year: {(school_years['Years active'] == 1).sum()}")
print(f"Schools in 2-3 years: {((school_years['Years active'] >= 2) & (school_years['Years active'] <= 3)).sum()}")
print(f"Schools in 4+ years: {(school_years['Years active'] >= 4).sum()}")
print(f"Schools in all 7 years: {(school_years['Years active'] == 7).sum()}")

print("\n" + "="*70)

## 6. Comparative Data Visualization

In [ ]:
# Regional heatmap: Expenditure by region and year
print("Creating regional expenditure heatmap...")
region_year_matrix = siope_merged.pivot_table(
    values='importo_euro',
    index='codice_regione',
    columns='anno',
    aggfunc='sum'
) / 1e6  # Convert to millions

# Get top 10 regions by total expenditure
top_regions = region_year_matrix.sum(axis=1).nlargest(10).index
region_year_top = region_year_matrix.loc[top_regions]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(region_year_top, annot=True, fmt='.0f', cmap='YlGn', ax=ax, cbar_kws={'label': '€ Millions'})
ax.set_title('School Expenditure by Region and Year (Top 10 Regions)\n€ Millions', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Region Code')
plt.tight_layout()
plt.show()

# Regional distribution: Schools vs Expenditure
print("\nCreating regional distribution comparison...")
regional_stats = siope_merged.groupby('codice_regione').agg({
    'codice_ente': 'nunique',
    'importo_euro': 'sum'
}).reset_index()
regional_stats.columns = ['Region', 'Schools', 'Total expenditure']
regional_stats = regional_stats.sort_values('Total expenditure', ascending=False).head(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Schools by region
ax1.barh(regional_stats['Region'].astype(str), regional_stats['Schools'], color='steelblue', alpha=0.7)
ax1.set_xlabel('Number of Schools')
ax1.set_title('Schools by Region (Top 15)')
ax1.grid(True, alpha=0.3, axis='x')

# Expenditure by region
ax2.barh(regional_stats['Region'].astype(str), regional_stats['Total expenditure']/1e6, color='darkgreen', alpha=0.7)
ax2.set_xlabel('Total Expenditure (€ Millions)')
ax2.set_title('Total Expenditure by Region (Top 15)')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Box plot: Expenditure distribution by year
print("\nCreating expenditure distribution comparison...")
fig, ax = plt.subplots(figsize=(12, 6))
siope_merged[siope_merged['importo_euro'] > 0].boxplot(column='importo_euro', by='anno', ax=ax)
ax.set_xlabel('Year')
ax.set_ylabel('Expenditure per Record (€)')
ax.set_title('Expenditure Distribution by Year (excluding zeros)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x/1000:.0f}K'))
plt.suptitle('')
plt.tight_layout()
plt.show()

## 7. Statistical Correlation and Summary Analysis

In [ ]:
# Correlation analysis at school level
print("SCHOOL-LEVEL CORRELATION ANALYSIS")
print("="*70)

# School summary
school_summary_analysis = siope_merged.groupby('codice_ente').agg({
    'importo_euro': ['sum', 'mean', 'count'],
    'anno': 'nunique',
    'codice_regione': 'first'
}).reset_index()
school_summary_analysis.columns = ['School code', 'Total expenditure', 'Mean per transaction', 'Transaction count', 'Years active', 'Region']

print(f"\nTotal schools: {len(school_summary_analysis)}")
print(f"\nSchool Expenditure Distribution:")
print(school_summary_analysis['Total expenditure'].describe())

# Top 10 schools by expenditure
print("\n\nTOP 10 SCHOOLS BY TOTAL EXPENDITURE")
top_schools = school_summary_analysis.nlargest(10, 'Total expenditure')[['School code', 'Total expenditure', 'Transaction count', 'Years active']]
print(top_schools.to_string(index=False))

# Correlation between years active and total expenditure
years_exp_corr = np.corrcoef(
    school_summary_analysis['Years active'],
    school_summary_analysis['Total expenditure']
)[0, 1]
print(f"\n\nCorrelation (Years Active vs Total Expenditure): {years_exp_corr:.3f}")

# Transaction frequency vs amount
freq_amount_corr = np.corrcoef(
    school_summary_analysis['Transaction count'],
    school_summary_analysis['Mean per transaction']
)[0, 1]
print(f"Correlation (Transaction Count vs Mean per Transaction): {freq_amount_corr:.3f}")

# Visualize correlations
print("\nCreating correlation visualizations...")
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# Scatter 1: Years active vs Total expenditure
ax1.scatter(school_summary_analysis['Years active'], school_summary_analysis['Total expenditure']/1e6, alpha=0.5)
ax1.set_xlabel('Years Active')
ax1.set_ylabel('Total Expenditure (€ Millions)')
ax1.set_title(f'Years Active vs Expenditure (r={years_exp_corr:.3f})')
ax1.grid(True, alpha=0.3)

# Scatter 2: Transaction count vs mean amount
ax2.scatter(school_summary_analysis['Transaction count'], school_summary_analysis['Mean per transaction'], alpha=0.5)
ax2.set_xlabel('Transaction Count')
ax2.set_ylabel('Mean per Transaction (€)')
ax2.set_title(f'Transaction Count vs Mean Amount (r={freq_amount_corr:.3f})')
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x/1000:.0f}K'))

# Distribution: Total expenditure per school
ax3.hist(school_summary_analysis['Total expenditure']/1e6, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Total Expenditure (€ Millions)')
ax3.set_ylabel('Number of Schools')
ax3.set_title('Distribution of School Expenditure')
ax3.grid(True, alpha=0.3, axis='y')

# Distribution: Years active
years_dist = school_summary_analysis['Years active'].value_counts().sort_index()
ax4.bar(years_dist.index, years_dist.values, color='darkgreen', alpha=0.7, edgecolor='black')
ax4.set_xlabel('Years Active')
ax4.set_ylabel('Number of Schools')
ax4.set_title('Distribution of Schools by Years Active')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Key findings summary
print("\n" + "="*70)
print("KEY FINDINGS SUMMARY")
print("="*70)
print(f"""
1. SIOPE COVERAGE (2020-2026):
   - {siope_merged['codice_ente'].nunique():,} schools with expenditure records
   - €{siope_merged['importo_euro'].sum():,.2f} total expenditure
   - {siope_merged['anno'].min()}-{siope_merged['anno'].max()} year range

2. TEMPORAL TRENDS:
   - YoY growth from 2020 (€{siope_merged[siope_merged['anno']==2020]['importo_euro'].sum():,.0f}) to 2025 (€{siope_merged[siope_merged['anno']==2025]['importo_euro'].sum():,.0f})
   - {(school_summary_analysis['Years active'] == 7).sum()} schools continuously reporting (all 7 years)
   - {(school_summary_analysis['Years active'] == 1).sum()} schools reporting only 1 year

3. TERRITORIAL DISTRIBUTION:
   - {siope_merged['codice_regione'].nunique()} regions covered
   - {siope_merged['codice_provincia'].nunique()} provinces covered

4. DATA QUALITY:
   - {len(negative_rows):,} records with negative amounts ({100*len(negative_rows)/len(siope_merged):.2f}%)
   - {len(schools_with_zero):,} records with zero expenditure ({100*len(schools_with_zero)/len(siope_merged):.2f}%)

5. CORRELATIONS:
   - Years Active ↔ Total Expenditure: {years_exp_corr:.3f}
   - Transaction Frequency ↔ Mean Amount: {freq_amount_corr:.3f}
""")
print("="*70)